# Batched Groq LLM Evaluation Runner

This notebook reads `data/processed/evaluation.json`, loads each `source_file`, sends batched PRs to Groq `gpt-oss-20b`, and appends raw model outputs to `outputs/llm_raw_responses.jsonl`.

In [2]:
import os, json, time
from dotenv import load_dotenv
import pandas as pd
from pathlib import Path
from groq import Groq

ROOT = Path("..").resolve()
EVAL_PATH = ROOT / 'data' / 'processed' / 'evaluation.json'
RAW_OUT = ROOT / 'outputs' / 'llm_raw_responses.txt'
PARSED_OUT = ROOT / 'outputs' / 'llm_reviews.json'
RAW_OUT.parent.mkdir(parents=True, exist_ok=True)

MODEL = 'openai/gpt-oss-20b'
BATCH_SIZE = 2
MAX_CHARS_PER_FILE = 8000

# evaluation.json range controls (inclusive, 0-based)
STARTING = 0
ENDING = 20

# Token controls to stay below Groq on_demand TPM limit
MODEL_CONTEXT_LIMIT = 8192  # conservative safe value
MAX_OUTPUT_TOKENS = 800

INPUT_TOKEN_BUDGET = int(MODEL_CONTEXT_LIMIT * 0.75) - MAX_OUTPUT_TOKENS
CHARS_PER_TOKEN = 2.5  # safer for code

load_dotenv()  # Load .env file if present

GROQ_API_KEY = os.environ['GROQ_API_KEY']
GROQ_API_URL = os.environ.get('GROQ_API_URL', 'https://api.groq.com')
client = Groq(api_key=GROQ_API_KEY, base_url=GROQ_API_URL)

In [3]:
def load_evaluation(path: Path):
    data = json.loads(path.read_text(encoding='utf-8'))
    return [r for r in data if isinstance(r, dict) and r.get('source_file')]

def read_source_text(source_file: str):
    p = ROOT / source_file
    if not p.exists():
        p = ROOT / 'data' / 'processed' / source_file
    text = p.read_text(encoding='utf-8')
    if len(text) <= MAX_CHARS_PER_FILE:
        return text
    half = MAX_CHARS_PER_FILE // 2
    return text[:half] + '\n\n...TRUNCATED...\n\n' + text[-half:]

def est_tokens(text: str):
    return max(1, len(text) // CHARS_PER_TOKEN)

records = load_evaluation(EVAL_PATH)
records = records[STARTING:ENDING + 1]
print('selected_records:', len(records), 'range:', STARTING, 'to', ENDING, '(inclusive)')

selected_records: 21 range: 0 to 20 (inclusive)


In [4]:
rows = []
row_idx = 1
for entry in records:
    pr_id = entry['id']
    for review in entry.get('ground_truth_reviews', []):
        rows.append({
            'id': f"row_{row_idx}" ,
            'PR': pr_id,
            'line_number': review.get('line_number'),
            'violation': review.get('violation_category'),
            'review_comment': review.get('review_comment')
        })
        row_idx += 1

ground_truth_df = pd.DataFrame(rows, columns=['id', 'PR', 'line_number', 'violation', 'review_comment'])
print('ground_truth_df shape:', ground_truth_df.shape)
ground_truth_df.head(10)

ground_truth_df shape: (126, 5)


,id,PR,line_number,violation,review_comment
0,row_1,synthetic-django_PR_21,9,unused_import,Unused import: os
1,row_2,synthetic-django_PR_21,10,unused_import,Unused import: sys
2,row_3,synthetic-django_PR_21,11,unused_import,Unused import: re
3,row_4,synthetic-django_PR_22,43,naming_convention,camelCase function name: cleanTitle
4,row_5,synthetic-django_PR_22,76,naming_convention,camelCase function name: cleanName
5,row_6,synthetic-django_PR_23,11,unused_import,Unused import: os
6,row_7,synthetic-django_PR_23,12,unused_import,Unused import: sys
7,row_8,synthetic-django_PR_23,13,unused_import,Unused import: re
8,row_9,synthetic-django_PR_23,24,indentation,Non-4-space indent (2 spaces)
9,row_10,synthetic-django_PR_23,34,indentation,Non-4-space indent (6 spaces)


In [5]:
def build_prompt(batch):
    header = """You are reviewing Python PR files for a code-quality benchmark dataset.

You may receive a BATCH containing multiple PR items in one request.
Each PR item is independent and must produce exactly one output object.
You will receive PR data in the following format:
- PR id
- Python source file content

Project-specific violation taxonomy in this dataset:
- naming_convention: camelCase names in Python (functions/params/variables/attributes)
- indentation: non-4-space indentation or inconsistent block indentation
- unused_import: imported symbol/module never used
- mutable_default: default [] / {} in function parameters
- documentation_formatting: docstring indentation/format mismatch

Generation rules (strict):
1) Return ONLY valid JSON array. No prose, no markdown.
2) One output object per PR using this exact key: PR_ID.
3) Detect likely violations directly from source code; do not assume any pre-annotated issues.
4) Include only findings that fit the allowed taxonomy categories.
5) For each finding, provide precise line_number, violation_category, and review_comment.
6) Return up to 5 strongest findings per PR item.
7) review_comment style: short, concrete, technical; mention the exact issue and preferred fix.

Output schema (STRICT, exact keys):
[
  {
    "PR_ID": <string>,
    "llm_reviews": [
      {"line_number": <int>, "violation_category": <string>, "review_comment": <string>}
    ]
  }
]
Return ONLY this JSON array structure, with one object per input PR item.
"""

    blocks = [header]
    for item in batch:
        blocks.append(f"""
PR_ITEM_START
id: {item['id']}
source_file: {item['source_file']}
source_code:
```python
{item['source_code']}
```
PR_ITEM_END
""")
    return '\n'.join(blocks)

In [6]:
def call_groq(prompt):
    response = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        messages=[
            {'role': 'user', 'content': prompt}
        ]
    )
    return response

In [ ]:
# Single-sample Groq run: pick the first entry and print response (no file writes)
sample = records[0]
entry = {
    'id': sample['id'],
    'repo': sample.get('repo'),
    'source_file': sample['source_file'],
    'source_code': read_source_text(sample['source_file'])
}
prompt = build_prompt([entry])
print('=== PROMPT (truncated 2000 chars) ===')
print(prompt[:2000])
print('\n=== CALLING GROQ ===')
resp = call_groq(prompt)
content = resp.choices[0].message.content
print('\n=== RESPONSE ===')
print(content)

=== PROMPT (truncated 2000 chars) ===
You are reviewing Python PR files for a code-quality benchmark dataset.

You may receive a BATCH containing multiple PR items in one request.
Each PR item is independent and must produce exactly one output object.
You will receive PR data in the following format:
- PR id
- Python source file content

Project-specific violation taxonomy in this dataset:
- naming_convention: camelCase names in Python (functions/params/variables/attributes)
- indentation: non-4-space indentation or inconsistent block indentation
- unused_import: imported symbol/module never used
- mutable_default: default [] / {} in function parameters
- documentation_formatting: docstring indentation/format mismatch

Generation rules (strict):
1) Return ONLY valid JSON array. No prose, no markdown.
2) One output object per PR using this exact key: PR_ID.
3) Detect likely violations directly from source code; do not assume any pre-annotated issues.
4) Include only findings that fit th

In [8]:
records

[{'id': 'synthetic-django_PR_21',
  'repo': 'kannan-dedsec/synthetic-django',
  'source_path': 'admin.py',
  'source_file': 'evaluation_files/synthetic-django_PR_21_admin.py',
  'ground_truth_reviews': [{'line_number': 9,
    'violation_category': 'unused_import',
    'review_comment': 'Unused import: os'},
   {'line_number': 10,
    'violation_category': 'unused_import',
    'review_comment': 'Unused import: sys'},
   {'line_number': 11,
    'violation_category': 'unused_import',
    'review_comment': 'Unused import: re'}]},
 {'id': 'synthetic-django_PR_22',
  'repo': 'kannan-dedsec/synthetic-django',
  'source_path': 'forms.py',
  'source_file': 'evaluation_files/synthetic-django_PR_22_forms.py',
  'ground_truth_reviews': [{'line_number': 43,
    'violation_category': 'naming_convention',
    'review_comment': 'camelCase function name: cleanTitle'},
   {'line_number': 76,
    'violation_category': 'naming_convention',
    'review_comment': 'camelCase function name: cleanName'}]},
 {'

In [10]:
batch_size=BATCH_SIZE
dry_run=False
prepared = []
for r in records:
    prepared.append({
        'id': r['id'],
        'repo': r['repo'],
        'source_file': r['source_file'],
        'source_code': read_source_text(r['source_file'])
    })

id_to_repo = {x['id']: x['repo'] for x in prepared}
i = 0
batch_no = 0
while i < 4:
    batch = []

    while i < len(prepared) and len(batch) < BATCH_SIZE:
        candidate = prepared[i]
        batch.append(candidate)
        i += 1

    batch_no += 1
    prompt = build_prompt(batch)

    resp = call_groq(prompt)
    print("Hi ",resp)
    choice0 = resp.choices[0]
    message0 = choice0.message
    content = message0.content

    # Print response diagnostics before any file write
    print(f'\n=== BATCH {batch_no} RESPONSE DIAGNOSTICS ===')
    print(f'PR ids: {[p["id"] for p in batch]}')
    print(f'content_type: {type(content).__name__}')
    if isinstance(content, str):
        print(f'content_len: {len(content)}')
        print('response_preview:')
        print(content[:1200])
    else:
        print('response_preview_non_str:')
        print(content)

    content_to_write = content if isinstance(content, str) else str(content)

    # Append raw response content with batch metadata
    with open(RAW_OUT, 'a', encoding='utf-8') as rf:
        rf.write(f'Batch {batch_no} - PR ids: {[p["id"] for p in batch]}\n')
        rf.write(f'repos: {[id_to_repo[p["id"]] for p in batch]}\n')
        rf.write(content_to_write)
        rf.write('\n\n\n')

    print(f"Batch {batch_no}: {len(batch)} PRs, est_input_tokens={est_tokens(prompt)}")

Hi  ChatCompletion(id='chatcmpl-d6604343-d6c8-4d93-8714-6bf2c3df5d4f', choices=[Choice(finish_reason='length', index=0, logprobs=None, message=ChatCompletionMessage(content='', role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning='We need to produce JSON array with two objects, each with PR_ID and llm_reviews list of findings up to 5. We must detect violations from taxonomy: naming_convention (camelCase names in Python functions/params/variables/attributes), indentation (non-4-space or inconsistent), unused_import, mutable_default, documentation_formatting (docstring indentation/format mismatch).\n\nWe need to analyze each file.\n\nFirst file synthetic-django_PR_21_admin.py.\n\nImports: os, sys, re are unused. So unused_import violations at lines where they are imported. Let\'s find line numbers. The file starts with triple quotes docstring lines 1-? Let\'s count.\n\nLine numbers:\n\n1: """ \n2: Admin configuration for the blog application.\n3: \n4: Th

In [ ]:
batch_size=BATCH_SIZE
dry_run=False
prepared = []
for r in records:
    prepared.append({
        'id': r['id'],
        'repo': r['repo'],
        'source_file': r['source_file'],
        'source_code': read_source_text(r['source_file'])
    })

id_to_repo = {x['id']: x['repo'] for x in prepared}
i = 0
batch_no = 0
while i < len(prepared):
    batch = []
    current_tokens = 0

    while i < len(prepared) and len(batch) < BATCH_SIZE:
        candidate = prepared[i]
        candidate_prompt = build_prompt([candidate])
        candidate_tokens = est_tokens(candidate_prompt)

        if candidate_tokens > INPUT_TOKEN_BUDGET:
            print(f"Skipping oversized entry: {candidate['id']}")
            i += 1
            continue

        if current_tokens + candidate_tokens > INPUT_TOKEN_BUDGET:
            break

        batch.append(candidate)
        current_tokens += candidate_tokens
        i += 1

    batch_no += 1
    prompt = build_prompt(batch)

    resp = call_groq(prompt)
    choice0 = resp.choices[0]
    message0 = choice0.message
    content = message0.content

    # Print response diagnostics before any file write
    print(f'\n=== BATCH {batch_no} RESPONSE DIAGNOSTICS ===')
    print(f'PR ids: {[p["id"] for p in batch]}')
    print(f'est_input_tokens: {est_tokens(prompt)}')
    print(f'finish_reason: {choice0.finish_reason}')
    print(f'usage: {resp.usage}')
    print(f'content_type: {type(content).__name__}')
    if isinstance(content, str):
        print(f'content_len: {len(content)}')
        print(f'is_empty_after_strip: {len(content.strip()) == 0}')
        print('response_preview:')
        print(content[:1200])
    else:
        print('response_preview_non_str:')
        print(content)

    content_to_write = content if isinstance(content, str) else str(content)

    # Append raw response content with batch metadata
    with open(RAW_OUT, 'a', encoding='utf-8') as rf:
        rf.write(f'Batch {batch_no} - PR ids: {[p["id"] for p in batch]}\n')
        rf.write(f'repos: {[id_to_repo[p["id"]] for p in batch]}\n')
        rf.write(content_to_write)
        rf.write('\n\n\n')

    print(f"batch {batch_no}: {len(batch)} PRs, est_input_tokens={est_tokens(prompt)}")

In [ ]:
ground_truth_df[ground_truth_df['PR'] == "synthetic-django_PR_21"]

## Usage

1. Install library: `pip install groq`
2. Export key: `GROQ_API_KEY`
3. In the first code cell, set `STARTING` and `ENDING` (inclusive, 0-based index in `evaluation.json`).
4. Tune token controls in the first code cell: `MAX_OUTPUT_TOKENS` and `INPUT_TOKEN_BUDGET`.
5. Keep `dry_run=True` for prompt inspection, then set `dry_run=False` for real execution.
6. Raw outputs append to `outputs/llm_raw_responses.txt` with PR metadata headers.
7. Parsed outputs are incrementally written to `outputs/llm_reviews.json` after each parsed batch.